# GPT-2 stats analysis

This notebook computes:
- Embedding covariance and eigen spectrum
- Effective rank (99% energy) for Q, K, V, and out projection weights
- Hidden state covariance and eigen spectrum after parameter-free LayerNorm
- Mean log eigenvalue per layer and plots

Results are saved under results/<model>.

## 1. Imports and dependencies

This cell imports PyTorch, Transformers, Datasets, plotting, and small utilities used in the rest of the notebook.

In [1]:
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

c:\Users\Kelvin\Documents\GitHub\AI-practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

Adjust model names, dataset, sequence length, batch size, number of batches, and output directory here. Use `embedding_sample_size` to speed up embedding covariance if needed.

In [4]:
@dataclass
class Config:
    model_names = ["gpt2", "gpt2-large"]
    dataset_name = "Salesforce/wikitext"
    dataset_config = "wikitext-2-raw-v1"
    max_length = 128
    batch_size = 8
    num_batches = 8
    embedding_sample_size = None  # set an int to speed up embedding covariance
    cov_dtype = torch.float32
    eps = 1e-6
    output_dir = Path("results")


cfg = Config()
cfg.output_dir.mkdir(exist_ok=True)


## 3. Helper functions

Defines the parameter-free LayerNorm, covariance accumulation, eigenvalue sorting/plotting, effective rank computation, and batch construction.

In [3]:
def layernorm_no_params(x: torch.Tensor, eps: float) -> torch.Tensor:
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps)


def maybe_sample_rows(x: torch.Tensor, max_rows: Optional[int], seed: int = 42) -> torch.Tensor:
    if not max_rows or x.shape[0] <= max_rows:
        return x
    gen = torch.Generator().manual_seed(seed)
    idx = torch.randperm(x.shape[0], generator=gen)[:max_rows]
    return x[idx]


def covariance_from_rows(x: torch.Tensor) -> torch.Tensor:
    x = x.to(torch.float32)
    x = x - x.mean(dim=0, keepdim=True)
    return (x.T @ x) / (x.shape[0] - 1)


class CovAccumulator:
    def __init__(self, dim: int, dtype: torch.dtype):
        self.dim = dim
        self.dtype = dtype
        self.count = 0
        self.sum = torch.zeros(dim, dtype=dtype)
        self.sum_xtx = torch.zeros(dim, dim, dtype=dtype)

    def update(self, x: torch.Tensor) -> None:
        x = x.to(self.dtype)
        self.sum += x.sum(dim=0)
        self.sum_xtx += x.T @ x
        self.count += x.shape[0]

    def covariance(self) -> torch.Tensor:
        if self.count < 2:
            raise ValueError("Not enough samples for covariance")
        mean = self.sum / self.count
        cov = (self.sum_xtx - self.count * torch.outer(mean, mean)) / (self.count - 1)
        return cov


def eigvals_sorted(cov: torch.Tensor, eps: float) -> torch.Tensor:
    eigvals = torch.linalg.eigvalsh(cov.double())
    eigvals = torch.clamp(eigvals, min=eps)
    return torch.flip(eigvals, dims=[0])


def plot_loglog_eigs(eigvals: torch.Tensor, out_path: Path, title: str) -> None:
    xs = torch.arange(1, eigvals.shape[0] + 1).numpy()
    ys = eigvals.cpu().numpy()
    plt.figure(figsize=(6, 4))
    plt.plot(xs, ys, linewidth=1.0)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Index")
    plt.ylabel("Eigenvalue")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


def effective_rank_99(weight: torch.Tensor) -> tuple[int, float]:
    s = torch.linalg.svdvals(weight.float())
    energy = torch.cumsum(s * s, dim=0)
    total = energy[-1]
    k = int(torch.searchsorted(energy, 0.99 * total).item() + 1)
    denom = min(weight.shape)
    return k, k / denom


def build_batches(tokenizer, texts, batch_size: int, max_length: int, num_batches: int):
    batches = []
    needed = batch_size * num_batches
    texts = texts[:needed]
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        if len(batch_texts) < batch_size:
            break
        enc = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )
        batches.append(enc)
    return batches

## 3b. Embedding covariance (GPT-2 vs GPT-2-large)

This optional cell computes the embedding covariance for each model and saves it to results/<model>/embedding_cov.pt. Use `embedding_sample_size` in the config to subsample rows if you want it faster.

In [8]:
def compute_embedding_covariances(cfg: Config):
    covs = {}
    for model_name in cfg.model_names:
        print(f"Embedding covariance: {model_name}")
        model = AutoModelForCausalLM.from_pretrained(model_name)
        wte = model.transformer.wte.weight.detach().cpu()
        wte = maybe_sample_rows(wte, cfg.embedding_sample_size)
        cov = covariance_from_rows(wte)

        out_dir = cfg.output_dir / model_name.replace("/", "_")
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "embedding_cov.pt"
        torch.save(cov, out_path)

        covs[model_name] = cov
        print(f"  cov shape: {tuple(cov.shape)} saved to {out_path}")
        del model

    return covs


embedding_covs = compute_embedding_covariances(cfg)


Embedding covariance: gpt2


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10213.35it/s]


  cov shape: (768, 768) saved to results\gpt2\embedding_cov.pt
Embedding covariance: gpt2-large


Loading weights: 100%|██████████| 436/436 [00:00<00:00, 9857.62it/s]


  cov shape: (1280, 1280) saved to results\gpt2-large\embedding_cov.pt


## 3c. Embedding covariance visualization

Plots the embedding covariance eigenvalue spectrum for GPT-2 and GPT-2-large and prints a short numeric summary.

In [9]:
def summarize_embedding_covariances(covs: dict[str, torch.Tensor], eps: float) -> None:
    for model_name, cov in covs.items():
        eigs = eigvals_sorted(cov, eps)
        logmean = float(torch.log(eigs).mean().item())
        top5 = [float(v) for v in eigs[:5].tolist()]
        print(f"{model_name}: log-eig mean={logmean:.4f}, top5={top5}")

        out_dir = cfg.output_dir / model_name.replace("/", "_")
        plot_loglog_eigs(eigs, out_dir / "embedding_eigs.png", f"{model_name} embedding eigs")


covs = embedding_covs if "embedding_covs" in globals() else {}
if not covs:
    covs = {}
    for model_name in cfg.model_names:
        out_dir = cfg.output_dir / model_name.replace("/", "_")
        cov_path = out_dir / "embedding_cov.pt"
        covs[model_name] = torch.load(cov_path)

summarize_embedding_covariances(covs, cfg.eps)


gpt2: log-eig mean=-4.5100, top5=[0.21092021775719866, 0.09588375776950425, 0.08349939015484492, 0.08004664425675076, 0.07619011486362545]
gpt2-large: log-eig mean=-6.1150, top5=[0.08969203719718621, 0.04228335953638964, 0.03770735389233164, 0.027592958076727066, 0.023788621725481462]


## 4. Load data and build batches

Loads Wikitext-2, filters empty lines, tokenizes, and prepares a fixed number of batches for repeatable statistics.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(cfg.dataset_name, cfg.dataset_config, split="train")
texts = [t for t in dataset["text"] if t.strip()]
batches = build_batches(tokenizer, texts, cfg.batch_size, cfg.max_length, cfg.num_batches)
print(f"Prepared {len(batches)} batches")

c:\Users\Kelvin\Documents\GitHub\AI-practice\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kelvin\.cache\huggingface\hub\datasets--Salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 935939.65 examples/s]


Prepared 8 batches


## 5. Per-model analysis

Runs all requested statistics for one model: embedding covariance/eigen spectrum, QKV/out projection effective rank, hidden-state covariance after parameter-free LayerNorm, per-layer eigen plots, and log-eigenvalue means.

In [6]:
def analyze_model(model_name: str, tokenizer, batches, cfg: Config, device: torch.device) -> None:
    print(f"Loading model: {model_name}")
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.to(device)
    model.eval()

    model_dir = cfg.output_dir / model_name.replace("/", "_")
    model_dir.mkdir(parents=True, exist_ok=True)

    wte = model.transformer.wte.weight.detach().cpu()
    wte = maybe_sample_rows(wte, cfg.embedding_sample_size)
    cov_emb = covariance_from_rows(wte)
    eig_emb = eigvals_sorted(cov_emb, cfg.eps)
    plot_loglog_eigs(eig_emb, model_dir / "embedding_eigs.png", f"{model_name} embedding eigs")
    emb_logmean = float(torch.log(eig_emb).mean().item())

    with open(model_dir / "embedding_stats.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["log_eig_mean"])
        writer.writerow([emb_logmean])

    ranks = {"q": [], "k": [], "v": [], "o": []}
    for block in tqdm(model.transformer.h, desc=f"SVD ranks ({model_name})"):
        attn = block.attn
        d = model.config.n_embd
        w_qkv = attn.c_attn.weight.detach().cpu()
        w_q = w_qkv[:, :d]
        w_k = w_qkv[:, d : 2 * d]
        w_v = w_qkv[:, 2 * d :]
        w_o = attn.c_proj.weight.detach().cpu()

        _, rq = effective_rank_99(w_q)
        _, rk = effective_rank_99(w_k)
        _, rv = effective_rank_99(w_v)
        _, ro = effective_rank_99(w_o)

        ranks["q"].append(rq)
        ranks["k"].append(rk)
        ranks["v"].append(rv)
        ranks["o"].append(ro)

    with open(model_dir / "effective_ranks.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["layer", "q", "k", "v", "o"])
        for i in range(len(ranks["q"])):
            writer.writerow([i, ranks["q"][i], ranks["k"][i], ranks["v"][i], ranks["o"][i]])

    plt.figure(figsize=(7, 4))
    layers = list(range(len(ranks["q"])))
    plt.plot(layers, ranks["q"], label="Q")
    plt.plot(layers, ranks["k"], label="K")
    plt.plot(layers, ranks["v"], label="V")
    plt.plot(layers, ranks["o"], label="Out")
    plt.xlabel("Layer")
    plt.ylabel("Normalized effective rank (99%)")
    plt.title(f"{model_name} effective rank vs depth")
    plt.legend()
    plt.tight_layout()
    plt.savefig(model_dir / "effective_rank_vs_depth.png")
    plt.close()

    num_layers = model.config.n_layer
    accs = [CovAccumulator(model.config.n_embd, cfg.cov_dtype) for _ in range(num_layers)]

    for batch in tqdm(batches, desc=f"Hidden states ({model_name})"):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch, output_hidden_states=True, use_cache=False)
        hidden_states = outputs.hidden_states[1:]

        for i, h in enumerate(hidden_states):
            x = h.reshape(-1, h.shape[-1])
            x = layernorm_no_params(x, cfg.eps)
            accs[i].update(x.cpu())

    logmeans = []
    for i, acc in tqdm(list(enumerate(accs)), desc=f"Eig spectra ({model_name})"):
        cov = acc.covariance()
        eigs = eigvals_sorted(cov, cfg.eps)
        plot_loglog_eigs(
            eigs,
            model_dir / f"layer_{i:02d}_eigs.png",
            f"{model_name} layer {i} eigs",
        )
        logmeans.append(float(torch.log(eigs).mean().item()))

    with open(model_dir / "layer_log_eig_mean.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["layer", "log_eig_mean"])
        for i, v in enumerate(logmeans):
            writer.writerow([i, v])

    plt.figure(figsize=(7, 4))
    plt.plot(list(range(len(logmeans))), logmeans, marker="o", linewidth=1.0)
    plt.xlabel("Layer")
    plt.ylabel("Mean log eigenvalue")
    plt.title(f"{model_name} log-eig mean vs depth")
    plt.tight_layout()
    plt.savefig(model_dir / "log_eig_mean_vs_depth.png")
    plt.close()

## 6. Run the pipeline

Selects CPU/GPU and executes the analysis for GPT-2 and GPT-2-large.

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

for model_name in cfg.model_names:
    analyze_model(model_name, tokenizer, batches, cfg, device)

print("Done. See results/<model> for outputs.")

Device: cpu
Loading model: gpt2


c:\Users\Kelvin\Documents\GitHub\AI-practice\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kelvin\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Eig spectra (gpt2): 100%|██████████| 12/12 [00:04<00:00,  2.44it/s]


Loading model: gpt2-large


c:\Users\Kelvin\Documents\GitHub\AI-practice\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Kelvin\.cache\huggingface\hub\models--gpt2-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Eig spectra (gpt2-large): 100%|██████████| 36/36 [00:17<00:00,  2.00it/s]

Done. See results/<model> for outputs.
